# Day 4 — Session 4 Support Notebook
## Unsupervised Learning for Discovery

**Session goal:** Use unlabeled engineering measurements to identify possible operating regimes, visualize high-dimensional structure, and evaluate whether discovered groups are useful.

This notebook adapts patterns used in the machine learning course:

- pandas DataFrames for loading and inspection;
- Matplotlib for visualization;
- standardization learned from the available feature data;
- scikit-learn transformers and estimators;
- pipeline-style, repeatable analysis.

The notebook introduces:

- k-means clustering;
- hierarchical clustering;
- DBSCAN;
- principal component analysis;
- silhouette analysis;
- cluster profiles;
- post-hoc domain interpretation;
- limitations of unsupervised results.

## Learning outcomes

By the end of the notebook, faculty should be able to:

1. Explain how unsupervised learning differs from supervised learning.
2. Prepare numerical features for distance-based clustering.
3. Apply k-means, hierarchical clustering, and DBSCAN.
4. Use PCA to visualize multivariable data.
5. Interpret silhouette scores and cluster profiles.
6. Explain why a cluster is not automatically a scientifically meaningful category.

## 1. Load the engineering dataset

Each row represents one process run.

The column `known_operating_condition` is **reference metadata** created for this workshop. It is not used to train the clustering algorithms. It will be used only at the end for post-hoc interpretation.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv("engineering_operating_regimes.csv")

print("Dataset shape:", data.shape)
data.head()

## 2. Define the unsupervised-learning question

**Question:** Do the process measurements contain natural groupings that may correspond to distinct operating regimes?

- Unit of observation: one process run
- Inputs: six numerical process and quality measurements
- Target: none
- Output: cluster membership or noise designation
- Validation: geometry, stability, domain meaning, and optional reference comparison

In [ ]:
feature_columns = [
    "temperature_c",
    "pressure_kpa",
    "speed_mm_s",
    "vibration_mm_s",
    "energy_kwh",
    "surface_roughness_um"
]

X = data[feature_columns].copy()

problem_summary = {
    "Unit of observation": "One process run",
    "Inputs": feature_columns,
    "Target": "None",
    "Task": "Unsupervised clustering",
    "Possible outputs": "Cluster labels or noise points",
    "Primary interpretation": "Operating-regime discovery"
}

problem_summary

## 3. Inspect scale and missing values

Distance-based methods are sensitive to feature scale.

A variable measured in hundreds, such as pressure, can dominate a variable measured in single digits, such as vibration, unless the features are standardized.

In [ ]:
audit = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing_count": X.isna().sum(),
    "mean": X.mean(),
    "standard_deviation": X.std(),
    "minimum": X.min(),
    "maximum": X.max()
})

audit.round(3)

## 4. Visualize two raw features

Two-dimensional plots are useful, but they show only part of the six-dimensional feature space.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(data["pressure_kpa"], data["vibration_mm_s"])
plt.xlabel("Pressure (kPa)")
plt.ylabel("Vibration (mm/s)")
plt.title("Raw process measurements")
plt.show()

## 5. Impute and standardize the features

The machine learning course emphasized that scaling parameters should be learned and then reused consistently.

Here, preprocessing is represented as a scikit-learn pipeline:

1. replace missing numerical values with the median;
2. standardize each feature to approximately zero mean and unit variance.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

preparation = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_scaled = preparation.fit_transform(X)

scaled_summary = pd.DataFrame(
    X_scaled,
    columns=feature_columns
).agg(["mean", "std"]).T

scaled_summary.round(3)

## 6. Use PCA for two-dimensional visualization

PCA creates new components that summarize directions of variation in the standardized feature space.

PCA is used here for visualization—not as proof that only two dimensions are sufficient for every analysis.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_table = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1]
})

print("Explained variance by PC1 and PC2:",
      np.round(pca.explained_variance_ratio_, 3))
print("Total variance shown:",
      round(pca.explained_variance_ratio_.sum(), 3))

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(pca_table["PC1"], pca_table["PC2"])
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("PCA view of standardized process runs")
plt.show()

## 7. Explore the number of k-means clusters

K-means requires the number of clusters in advance.

Two common diagnostics are:

- **inertia:** within-cluster squared distance;
- **silhouette score:** compares within-cluster cohesion with separation from other clusters.

These metrics support a decision, but do not determine scientific meaning.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

diagnostics = []

for k in range(2, 7):
    model = KMeans(
        n_clusters=k,
        n_init=20,
        random_state=42
    )
    labels = model.fit_predict(X_scaled)

    diagnostics.append({
        "number_of_clusters": k,
        "inertia": model.inertia_,
        "silhouette_score": silhouette_score(X_scaled, labels)
    })

diagnostics = pd.DataFrame(diagnostics)
diagnostics.round(3)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    diagnostics["number_of_clusters"],
    diagnostics["silhouette_score"],
    marker="o"
)
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.title("K-means cluster-separation diagnostic")
plt.show()

## 8. Fit k-means with three clusters

Cluster labels such as 0, 1, and 2 are arbitrary identifiers. Their numerical order has no disciplinary meaning.

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    n_init=20,
    random_state=42
)

kmeans_labels = kmeans.fit_predict(X_scaled)
data["kmeans_cluster"] = kmeans_labels

print("K-means silhouette score:",
      round(silhouette_score(X_scaled, kmeans_labels), 3))
print("Cluster sizes:")
display(data["kmeans_cluster"].value_counts().sort_index().to_frame("count"))

In [ ]:
plt.figure(figsize=(7, 5))

for cluster_id in sorted(data["kmeans_cluster"].unique()):
    mask = data["kmeans_cluster"] == cluster_id
    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        label=f"Cluster {cluster_id}"
    )

plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("K-means clusters shown in PCA space")
plt.legend()
plt.show()

## 9. Interpret k-means using cluster profiles

Cluster profiles describe the average original-unit measurements within each group.

Domain experts should ask whether the profiles correspond to plausible operating conditions.

In [ ]:
kmeans_profiles = (
    data.groupby("kmeans_cluster")[feature_columns]
    .mean()
    .round(2)
)

kmeans_profiles

## 10. Apply hierarchical clustering

Agglomerative hierarchical clustering begins with individual observations and repeatedly merges the closest groups.

The dendrogram provides a visual summary of the merge structure.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram

hierarchical = AgglomerativeClustering(n_clusters=3)
hierarchical_labels = hierarchical.fit_predict(X_scaled)

data["hierarchical_cluster"] = hierarchical_labels

print("Hierarchical silhouette score:",
      round(silhouette_score(X_scaled, hierarchical_labels), 3))

In [ ]:
linkage_matrix = linkage(X_scaled, method="ward")

plt.figure(figsize=(10, 5))
dendrogram(
    linkage_matrix,
    truncate_mode="level",
    p=4,
    no_labels=True
)
plt.xlabel("Merged observations or groups")
plt.ylabel("Ward distance")
plt.title("Truncated hierarchical-clustering dendrogram")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

for cluster_id in sorted(data["hierarchical_cluster"].unique()):
    mask = data["hierarchical_cluster"] == cluster_id
    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        label=f"Cluster {cluster_id}"
    )

plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("Hierarchical clusters shown in PCA space")
plt.legend()
plt.show()

## 11. Apply DBSCAN

DBSCAN groups observations that form dense neighborhoods and labels isolated observations as noise.

Important parameters:

- `eps`: neighborhood radius;
- `min_samples`: minimum number of nearby observations required to form a dense region.

DBSCAN does not require the number of clusters in advance, but its result can be sensitive to parameter choice and feature scaling.

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(
    eps=0.95,
    min_samples=6
)

dbscan_labels = dbscan.fit_predict(X_scaled)
data["dbscan_cluster"] = dbscan_labels

print("DBSCAN labels and sizes:")
display(data["dbscan_cluster"].value_counts().sort_index().to_frame("count"))
print("Noise points:", int((dbscan_labels == -1).sum()))

In [ ]:
plt.figure(figsize=(7, 5))

for cluster_id in sorted(data["dbscan_cluster"].unique()):
    mask = data["dbscan_cluster"] == cluster_id
    label = "Noise" if cluster_id == -1 else f"Cluster {cluster_id}"
    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        label=label
    )

plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.title("DBSCAN groups and noise shown in PCA space")
plt.legend()
plt.show()

## 12. Compare the three clustering methods

There is no single universal clustering algorithm.

Compare:

- number and size of groups;
- silhouette score where appropriate;
- sensitivity to unusual observations;
- interpretability of cluster profiles;
- consistency with disciplinary expectations.

In [ ]:
comparison = pd.DataFrame([
    {
        "method": "K-means",
        "clusters": data["kmeans_cluster"].nunique(),
        "noise_points": 0,
        "silhouette": silhouette_score(X_scaled, kmeans_labels)
    },
    {
        "method": "Hierarchical",
        "clusters": data["hierarchical_cluster"].nunique(),
        "noise_points": 0,
        "silhouette": silhouette_score(X_scaled, hierarchical_labels)
    },
    {
        "method": "DBSCAN",
        "clusters": len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0),
        "noise_points": int((dbscan_labels == -1).sum()),
        "silhouette": (
            silhouette_score(
                X_scaled[dbscan_labels != -1],
                dbscan_labels[dbscan_labels != -1]
            )
            if len(set(dbscan_labels[dbscan_labels != -1])) > 1
            else np.nan
        )
    }
])

comparison.round(3)

## 13. Session synthesis

Key principles:

1. Unsupervised learning has inputs but no prediction target.
2. Scaling is essential for distance-based clustering.
3. PCA helps visualize structure but does not define the clusters.
4. K-means favors compact, approximately spherical groups.
5. Hierarchical clustering reveals nested merge structure.
6. DBSCAN can identify irregular groups and noise.
7. Silhouette score evaluates geometric separation, not scientific truth.
8. Cluster labels are arbitrary.
9. Domain interpretation and independent evidence are required.
10. A cluster should not automatically be treated as a new scientific category.

## Optional extensions

- Change the number of k-means clusters and compare profiles.
- Vary DBSCAN `eps` and `min_samples`.
- Remove the unusual runs and repeat the analysis.
- Run clustering without standardization and inspect the change.
- Use more PCA components and inspect cumulative explained variance.
- Compare clusters across different feature subsets.
- Test cluster stability using resampled datasets.